# Xem video nhận diện từ đúng thành sai

Notebook phát lại các frame Penn Action thành video, so sánh pose YOLO gốc và pose lọc trung bình 5 frame. Danh sách mặc định gồm 35 video có ít nhất một seed đổi từ đúng sang sai khi **giữ nguyên classifier**.

Chạy lần lượt các ô, chọn video rồi nhấn **Tạo video**. Bộ điều khiển video cho phép tạm dừng, tua và phát lại. Video được tạo trong bộ nhớ tạm. Tốc độ mặc định 15 frame/giây là tốc độ xem, không khẳng định FPS gốc của Penn Action.

Đỏ là pose YOLO. Annotation và bbox GT có thể bật để tham khảo khi cần. Hình trái là pose gốc sau fixed swap, hình phải là pose đã lọc. Pose được vẽ khi confidence ≥ 0,3. Bbox GT hỗ trợ chọn người và phép lọc dùng frame tương lai. Chuyển đổi dự đoán không chứng minh nguyên nhân sai nhãn.

In [ ]:
import csv
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import cv2
import ipywidgets as widgets
import numpy as np
from IPython.display import Video, clear_output, display
from scipy.io import loadmat


def find_root():
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / "outputs/penn_prediction_inventory_20261123/video_summary.csv").exists():
            return directory
    raise FileNotFoundError(
        "Mở notebook trong dự án và kiểm tra tệp outputs/penn_prediction_inventory_20261123/video_summary.csv."
    )


ROOT = find_root()
sys.path.insert(0, str(ROOT / "src/gt_baselines"))

# Resolve local modules after locating the repository.
import compare_person_selection as base  # noqa: E402
import evaluate_pose_corrections as correction  # noqa: E402
from previous_joint_visual_pilot_20261113 import render  # noqa: E402

INVENTORY = ROOT / "outputs/penn_prediction_inventory_20261123"
POSES = ROOT / "outputs/confidence_confirmation_penn_20261105/test_pose/poses"
FRAMES = ROOT / "data/raw/Penn_Action/frames"
LABELS = ROOT / "data/raw/Penn_Action/labels"
FEATURES = ROOT / "outputs/pose_controls_20261115/features_mean5.npz"
FFMPEG = shutil.which("ffmpeg")
if FFMPEG is None:
    raise RuntimeError("Cần ffmpeg trong PATH để tạo video MP4. Hãy cài ffmpeg và mở lại kernel.")


def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


all_videos = read_csv(INVENTORY / "video_summary.csv")
paired = read_csv(INVENTORY / "paired_video_seed.csv")
cases = [
    row
    for row in all_videos
    if row["method"] == "mean5" and row["mode"] == "frozen" and int(row["correct_wrong_count"]) > 0
]
cases.sort(key=lambda row: (-int(row["correct_wrong_count"]), row["id"]))
if len(cases) != 35:
    raise ValueError(f"Cần kiểm tra lại inventory: tìm thấy {len(cases)} video.")
print(f"Đã tìm thấy {len(cases)} video. Dữ liệu: {ROOT}")

In [ ]:
def load_video_data(video_id):
    pose_path = POSES / f"{video_id}.npz"
    label_path = LABELS / f"{video_id}.mat"
    with np.load(pose_path) as data:
        xy, conf = data["xy"].copy(), data["conf"].copy()
    variants, _ = correction.make_variants(xy, conf)
    raw, confidence = variants["fixed_swap"]
    filtered = variants["mean5"][0]

    # Đối chiếu đầu vào đã nội suy thời gian với feature dùng trong thí nghiệm.
    with np.load(FEATURES) as features:
        matches = np.flatnonzero(features["test_ids"].astype(str) == video_id)
        if len(matches) != 1 or not np.array_equal(
            base.resample(filtered), features["test_x"][matches[0]]
        ):
            raise ValueError(f"Feature lọc không khớp artifact của video {video_id}.")

    annotation = loadmat(label_path)
    gt = np.stack((annotation["x"][:, 1:13], annotation["y"][:, 1:13]), axis=-1)
    visible = annotation["visibility"][:, 1:13] > 0
    bbox = np.full((len(raw), 4), np.nan)
    count = min(len(raw), len(annotation["bbox"]))
    bbox[:count] = annotation["bbox"][:count]
    if len(gt) != len(raw):
        raise ValueError(f"Số frame annotation và pose không khớp ở {video_id}.")
    frame_paths = [FRAMES / video_id / f"{index + 1:06d}.jpg" for index in range(len(raw))]
    missing = [path.name for path in frame_paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Video {video_id} thiếu frame: {missing[:5]}")
    return raw, filtered, confidence, gt, visible, bbox, frame_paths


def create_video(video_id, fps, show_gt=True):
    raw, filtered, confidence, gt, visible, bbox, frame_paths = load_video_data(video_id)
    first = cv2.imread(str(frame_paths[0]))
    if first is None:
        raise ValueError(f"Không đọc được {frame_paths[0]}")
    height, width = first.shape[:2]
    # Đảm bảo kích thước chẵn cho H.264 và giữ tỉ lệ ảnh gốc.
    output_width = min(width * 2, 1280)
    output_width -= output_width % 2
    output_height = int(height * output_width / (width * 2))
    output_height -= output_height % 2
    output_height = max(output_height, 2)
    flags = np.zeros(12, dtype=bool)

    with tempfile.TemporaryDirectory(prefix="penn_review_") as temp_dir:
        output_path = Path(temp_dir) / f"{video_id}.mp4"
        command = [
            FFMPEG,
            "-hide_banner",
            "-loglevel",
            "error",
            "-y",
            "-f",
            "rawvideo",
            "-pix_fmt",
            "bgr24",
            "-s",
            f"{output_width}x{output_height}",
            "-r",
            str(fps),
            "-i",
            "-",
            "-an",
            "-c:v",
            "libx264",
            "-preset",
            "veryfast",
            "-pix_fmt",
            "yuv420p",
            "-movflags",
            "+faststart",
            str(output_path),
        ]
        process = subprocess.Popen(command, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
        try:
            for index, frame_path in enumerate(frame_paths):
                frame = cv2.imread(str(frame_path))
                if frame is None or frame.shape != first.shape:
                    raise ValueError(f"Frame lỗi hoặc khác kích thước: {frame_path}")
                vis = visible[index] if show_gt else np.zeros(12, dtype=bool)
                box = bbox[index] if show_gt else np.full(4, np.nan)
                title = f"{video_id} | frame {index + 1}/{len(raw)}"
                left = render(
                    frame,
                    raw[index],
                    confidence[index],
                    gt[index],
                    vis,
                    box,
                    flags,
                    title + " | RAW",
                )
                right = render(
                    frame,
                    filtered[index],
                    confidence[index],
                    gt[index],
                    vis,
                    box,
                    flags,
                    title + " | MEAN 5",
                )
                image = cv2.resize(np.hstack((left, right)), (output_width, output_height))
                process.stdin.write(image.tobytes())
            process.stdin.close()
            error = process.stderr.read().decode("utf-8", errors="replace")
            if process.wait() != 0:
                raise RuntimeError(f"ffmpeg lỗi: {error}")
            video_bytes = output_path.read_bytes()
        finally:
            if process.poll() is None:
                process.kill()
                process.wait()
    return video_bytes, len(raw)


def predictions_for(video_id):
    return [
        row
        for row in paired
        if row["id"] == video_id and row["method"] == "mean5" and row["mode"] == "frozen"
    ]

In [ ]:
video_selector = widgets.Dropdown(
    options=[
        (
            f"{row['id']} — {row['action']} | đúng → sai: seed {row['correct_wrong_seeds']}",
            row["id"],
        )
        for row in cases
    ],
    description="Video:",
    layout=widgets.Layout(width="650px"),
)
fps_selector = widgets.IntSlider(value=15, min=1, max=30, description="FPS xem:")
gt_selector = widgets.Checkbox(value=False, description="Hiện annotation GT để tham khảo")
button = widgets.Button(description="Tạo video", button_style="primary", icon="play")
output = widgets.Output()


def show_selected(_=None):
    with output:
        clear_output(wait=True)
        video_id = video_selector.value
        rows = sorted(predictions_for(video_id), key=lambda row: int(row["seed"]))
        print(f"Video {video_id} — nhãn thật: {rows[0]['action']}")
        for row in rows:
            print(
                f"Seed {row['seed']}: {row['raw_label']} → {row['method_label']} ({row['transition']})"
            )
        print("Đang tạo video từ các frame...")
        try:
            video_bytes, frame_count = create_video(video_id, fps_selector.value, gt_selector.value)
            print(f"{frame_count} frame — {fps_selector.value} FPS xem")
            display(Video(video_bytes, embed=True, mimetype="video/mp4", width=1100))
        except Exception as error:
            print(f"Không tạo được video: {error}")
            raise


button.on_click(show_selected)
display(widgets.VBox([video_selector, fps_selector, gt_selector, button, output]))
show_selected()

Kết quả này là phân tích hồi cứu trên official test đã tham gia phát triển. Seed được liệt kê cho từng video; các seed còn lại có thể cho kết quả khác. Để đổi video hoặc FPS, chọn giá trị rồi nhấn **Tạo video** lần nữa.

## Phóng to các khớp thay đổi

Phần tùy chọn này dùng annotation GT làm tham chiếu để tìm khác biệt cục bộ. Chọn video ở phía trên, sau đó nhấn **Tìm điểm thay đổi**. Bảng xếp các khớp-frame theo mức tăng sai số so với GT. Chọn frame và khớp để xem vùng phóng to, cùng tọa độ gốc, sau lọc và annotation. Vòng tròn đỏ là pose gốc, xanh dương là pose sau lọc, xanh lá là GT; đường vàng nối hai vị trí gốc và sau lọc. Kết quả dựa trên các khớp có confidence ≥ 0,3 và GT visible, được chọn sau khi xem kết quả.

Các điểm này cho thấy sự thay đổi cục bộ, cần xem cả video và kiểm tra trái/phải hoặc người được chọn trước khi suy luận nguyên nhân mô hình dự đoán sai.

In [ ]:
from PIL import Image as PILImage

JOINTS = [
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]
diagnostic_button = widgets.Button(
    description="Tìm điểm thay đổi", button_style="info", icon="search"
)
frame_selector = widgets.IntSlider(
    description="Frame:",
    min=1,
    max=1,
    value=1,
    continuous_update=False,
    layout=widgets.Layout(width="650px"),
)
joint_selector = widgets.Dropdown(description="Khớp:", options=JOINTS, value="left_wrist")
diagnostic_output = widgets.Output()
detail_output = widgets.Output()
diagnostic_state = {}


def find_changes(_=None):
    with diagnostic_output:
        clear_output(wait=True)
        video_id = video_selector.value
        raw, filtered, confidence, gt, visible, bbox, frame_paths = load_video_data(video_id)
        with np.load(POSES / f"{video_id}.npz") as source:
            detected = source["status"] == 3
        eligible = (
            np.isfinite(raw).all(axis=2)
            & (raw > 0).all(axis=2)
            & np.isfinite(filtered).all(axis=2)
            & (filtered > 0).all(axis=2)
            & np.isfinite(confidence)
            & (confidence >= 0.3)
            & np.isfinite(gt).all(axis=2)
            & (gt > 0).all(axis=2)
            & visible
            & detected[:, None]
        )
        raw_error = np.linalg.norm(raw - gt, axis=2)
        filtered_error = np.linalg.norm(filtered - gt, axis=2)
        change = np.linalg.norm(filtered - raw, axis=2)
        delta = np.where(eligible, filtered_error - raw_error, np.nan)
        if not np.isfinite(delta).any():
            print(f"Video {video_id} không có điểm đủ điều kiện đối chiếu GT.")
            return
        diagnostic_state.clear()
        diagnostic_state.update(
            video_id=video_id,
            raw=raw,
            filtered=filtered,
            gt=gt,
            eligible=eligible,
            confidence=confidence,
            paths=frame_paths,
        )
        order = np.argsort(np.where(np.isfinite(delta), delta, -np.inf).ravel())[::-1]
        print(f"Video {video_id}: các điểm sai số tăng nhiều nhất so với GT (pixel)")
        print("Frame | Khớp | Lệch gốc → sau lọc | Khớp dịch chuyển")
        for index in order[:10]:
            frame, joint = np.unravel_index(index, delta.shape)
            if not np.isfinite(delta[frame, joint]):
                continue
            print(
                f"{frame + 1:>5} | {JOINTS[joint]:<15} | {raw_error[frame, joint]:6.1f} → {filtered_error[frame, joint]:6.1f} | {change[frame, joint]:6.1f}"
            )
        best_frame, best_joint = np.unravel_index(order[0], delta.shape)
        frame_selector.max = len(raw)
        frame_selector.value = int(best_frame + 1)
        joint_selector.value = JOINTS[best_joint]
        show_detail()


def show_detail(_=None):
    with detail_output:
        clear_output(wait=True)
        if not diagnostic_state:
            print("Nhấn Tìm điểm thay đổi để tải video đang chọn.")
            return
        state = diagnostic_state
        index, joint = frame_selector.value - 1, JOINTS.index(joint_selector.value)
        frame = cv2.imread(str(state["paths"][index]))
        if frame is None:
            raise FileNotFoundError(state["paths"][index])
        points = [state[name][index, joint] for name in ("raw", "filtered", "gt")]
        valid = [np.isfinite(p).all() and (p > 0).all() for p in points]
        if not any(valid):
            print("Không có tọa độ hợp lệ cho khớp này tại frame đã chọn.")
            return
        canvas = frame.copy()
        colors = [(0, 0, 255), (255, 100, 0), (0, 220, 0)]
        if valid[0] and valid[1]:
            cv2.line(
                canvas,
                tuple(np.rint(points[0]).astype(int)),
                tuple(np.rint(points[1]).astype(int)),
                (0, 255, 255),
                2,
                cv2.LINE_AA,
            )
        for point, ok, color in zip(points, valid, colors):
            if ok:
                cv2.circle(canvas, tuple(np.rint(point).astype(int)), 7, color, 2, cv2.LINE_AA)
        selected = np.array([p for p, ok in zip(points, valid) if ok])
        center = selected.mean(axis=0)
        half = max(60, int(np.ptp(selected, axis=0).max() / 2 + 35))
        height, width = frame.shape[:2]
        x0, x1 = max(0, int(center[0] - half)), min(width, int(center[0] + half))
        y0, y1 = max(0, int(center[1] - half)), min(height, int(center[1] + half))
        if x1 <= x0 or y1 <= y0:
            print("Tọa độ nằm ngoài ảnh, cần kiểm tra trực tiếp frame gốc.")
            return
        crop = canvas[y0:y1, x0:x1]
        scale = min(4.0, 650 / max(crop.shape[:2]))
        crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_NEAREST)
        print(f"Video {state['video_id']} | frame {index + 1} | {JOINTS[joint]}")
        print("Đỏ: gốc | Xanh dương: sau lọc | Xanh lá: GT | Vàng: thay đổi")
        if state["eligible"][index, joint]:
            before = np.linalg.norm(points[0] - points[2])
            after = np.linalg.norm(points[1] - points[2])
            print(
                f"Sai số so với GT: {before:.1f} → {after:.1f} px | thay đổi vị trí: {np.linalg.norm(points[1] - points[0]):.1f} px"
            )
        else:
            print("Điểm này không đủ điều kiện đối chiếu GT theo mask và confidence.")
        display(PILImage.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))


diagnostic_button.on_click(find_changes)
frame_selector.observe(show_detail, names="value")
joint_selector.observe(show_detail, names="value")
display(
    widgets.VBox(
        [diagnostic_button, diagnostic_output, frame_selector, joint_selector, detail_output]
    )
)